In [ ]:
import os
import gzip
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import scipy.io
from anndata import AnnData
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
sc.settings.set_figure_params(dpi=100)
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
#%load_ext rpy2.ipython
import sklearn.mixture


In [ ]:
import os

DATA_DIR       = os.environ.get('NPOD_DATA_DIR',      '/data')
REF_DIR        = os.environ.get('NPOD_REF_DIR',        '/reference_files')
PIPELINE_OUT   = os.environ.get('NPOD_PIPELINE_OUT',   os.path.join(DATA_DIR, 'snatac_pipeline_output'))
MULTIPLET_DIR  = os.environ.get('NPOD_MULTIPLET_DIR',  os.path.join(DATA_DIR, 'multiplet_output'))
CLEAN_DIR      = os.environ.get('NPOD_CLEAN_DIR',      DATA_DIR)

#EDIT: SAMPLE NAME
sample_name = 'MM_386'
qc = pd.read_table(os.path.join(DATA_DIR, '{}_metadata.txt'.format(sample_name)), sep='\t', header=0, index_col=0)
qc = qc[qc['unique_usable_reads'] > 5000]
qc = qc[qc['frac_promoters_used'] > 0.01]
qc = qc[qc['TSS.enrichment'] > 0.3]
qc

## Plot TSSe and unique usable counts post filtering

In [ ]:
#EDIT: PATH TO FILTERED BARCODES FILE WITH TSSE INFO AND OTHER QC METADATA  <-- create by quickly creating seurat objects for each sample and saving out data@meta.data
#EDIT: SAMPLE NAME
sample_name = 'MM_386'
metadata = '/data/'
qc = pd.read_table(os.path.join(metadata, '{}_metadata.txt'.format(sample_name)), sep='\t', header=0, index_col=0)
qc = qc[qc['unique_usable_reads'] > 5000]
qc = qc[qc['frac_promoters_used'] > 0.01]
qc = qc[qc['TSS.enrichment'] > 0.3]
qc

In [ ]:
#NOTE: plot TSS distribution
plt.style.use('seaborn-white')
fig, ax1 = plt.subplots(1,1,figsize=(5,5))
ax1.hist(qc['TSS.enrichment'], color='red', bins=np.arange(0,10,1), alpha=.5, label='_nolegend_')
ax1.axvline(2, color='green', label='0.5')

print("median TSSe is ") 
print(qc['TSS.enrichment'].mean())

In [ ]:
ex = pd.read_csv(os.path.join(MULTIPLET_DIR, '{}_excluded_barcodes.csv'.format(sample_name)))
exes = ex['Excluded Barcode'].tolist()
#NOTE: REMOVE THE '-1' FROM BARCODES BC CELLRANGER ADDS THAT BUT JOSH DOESN'T
exes2 = [x[:-2] for x in exes]

wd = os.path.join(PIPELINE_OUT, sample_name)
sp = scipy.io.mmread(os.path.join(wd, '{}.mtx.gz'.format(sample_name))).tocsr()
regions = open(os.path.join(wd, '{}.regions'.format(sample_name))).read().splitlines()

replace = open(os.path.join(wd, '{}.barcodes'.format(sample_name))).read().splitlines()
barcodes = qc.index.tolist()
#NOTE: REMOVE THE '-1' FROM BARCODES BC CELLRANGER ADDS THAT BUT JOSH DOESN'T
barcodes = [x[:-2] for x in barcodes]
print(len(replace))
print(len(barcodes))

# Removed excluded barcodes


In [ ]:
#EDIT: PATH TO MULTIPLET BARCODES
wd2 = 'data/multiplet_output/'

#check if _ is in file name
ex = pd.read_csv(os.path.join(wd2,'{}_excluded_barcodes.csv'.format(sample_name)))
# ex = pd.read_csv(os.path.join(wd2,'{}excluded_barcodes.csv'.format(sample_name)))
exes = ex['Excluded Barcode']
exes = exes.tolist()
#NOTE: REMOVE THE '-1' FROM BARCODES BC CELLRANGER ADDS THAT BUT JOSH DOESN'T
exes2 = [x[:-2] for x in exes]

#EDIT: PATH TO JOSH'S OUTPUT
wd = 'd{}'.format(sample_name)
sp = scipy.io.mmread(os.path.join(wd, '{}.mtx.gz'.format(sample_name))).tocsr()
regions = open(os.path.join(wd, '{}.regions'.format(sample_name))).read().splitlines()

#EDIT: PATH TO CLEANED BARCODES FILE
replace = open(os.path.join(wd, '{}.barcodes'.format(sample_name))).read().splitlines()
barcodes = qc.index.tolist()
#NOTE: REMOVE THE '-1' FROM BARCODES BC CELLRANGER ADDS THAT BUT JOSH DOESN'T
barcodes = [x[:-2] for x in barcodes]
print(len(replace))
print(len(barcodes))


In [ ]:
wd = os.path.join(PIPELINE_OUT, sample_name)
sp = scipy.io.mmread(os.path.join(wd, '{}.mtx.gz'.format(sample_name))).tocsr()
regions = open(os.path.join(wd, '{}.peaks'.format(sample_name))).read().splitlines()
barcodes = open(os.path.join(wd, '{}.barcodes'.format(sample_name))).read().splitlines()
adata = AnnData(sp, {'obs_names':barcodes}, {'var_names':regions})

#NOTE: Calculate highly variable genes/peaks
adata = AnnData(sp, {'obs_names':replace}, {'var_names':regions})
metrics = pd.read_table(os.path.join(PIPELINE_OUT, sample_name, '{}.qc_metrics.txt'.format(sample_name)), sep='\t', header=0, index_col=0)
adata.obs = adata.obs.join(metrics, how='inner')
adata = adata[(~adata.obs.index.isin(delete)),:].copy()

###This is the promoter list for hg38
promoters = pd.read_table(os.path.join(REF_DIR, 'promoter_list.txt'), sep='\t', header=None, index_col=0, names=['prom'])
promoter_names = promoters['prom'].to_dict()
adata.var.index = [promoter_names[b] if b in promoter_names else b for b in adata.var.index]
adata.var_names_make_unique(join='.')

adata.raw = sc.pp.log1p(adata, copy=True)
adata_orig = adata.copy()

sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
adata.var[sample_name] = (adata.raw.X > 0).sum(axis=0).A1
adata_filter = sc.pp.filter_genes_dispersion(adata.X, flavor='seurat', n_bins=100, min_disp=.25, min_mean=.01)
hvgs = adata.var.loc[adata_filter.gene_subset].index.tolist()
hvgs_check = adata.var.loc[adata.var.index.isin(hvgs)]
hvgs = hvgs_check.loc[(hvgs_check>0).sum(axis=1)==len(hvgs_check.columns)].index.tolist()
len(hvgs)

In [ ]:
wd = '/data/snatac_pipeline_output/'.format(sample_name)
sp = scipy.io.mmread(os.path.join(wd, '{}.mtx.gz'.format(sample_name))).tocsr()
regions = open(os.path.join(wd, '{}.peaks'.format(sample_name))).read().splitlines()
barcodes = open(os.path.join(wd, '{}.barcodes'.format(sample_name))).read().splitlines()
adata = AnnData(sp, {'obs_names':barcodes}, {'var_names':regions})



In [ ]:
#NOTE: Calculate highly variable genes/peaks
adata = AnnData(sp, {'obs_names':replace}, {'var_names':regions})
metrics = pd.read_table(os.path.join(wd, '{}.qc_metrics.txt'.format(sample_name)), sep='\t', header=0, index_col=0)
adata.obs = adata.obs.join(metrics, how='inner')
adata = adata[(~adata.obs.index.isin(delete)),:].copy()

###This is the promoter list for hg38
promoters = pd.read_table('/reference_files/promoter_list.txt', sep='\t', header=None, index_col=0, names=['prom'])
promoter_names = promoters['prom'].to_dict()
adata.var.index = [promoter_names[b] if b in promoter_names else b for b in adata.var.index]
adata.var_names_make_unique(join='.')

adata.raw = sc.pp.log1p(adata, copy=True)
adata_orig = adata.copy()

sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
adata.var[sample_name] = (adata.raw.X > 0).sum(axis=0).A1
adata_filter = sc.pp.filter_genes_dispersion(adata.X, flavor='seurat', n_bins=100, min_disp=.25, min_mean=.01)
hvgs = adata.var.loc[adata_filter.gene_subset].index.tolist()
hvgs_check = adata.var.loc[adata.var.index.isin(hvgs)]
hvgs = hvgs_check.loc[(hvgs_check>0).sum(axis=1)==len(hvgs_check.columns)].index.tolist()
len(hvgs)


# Cluster

In [ ]:
#NOTE: cluster time!!
adata = adata[:,adata.var.index.isin(hvgs)]
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)

adata.obs['log_usable_counts'] = np.log(adata_orig[:, adata_orig.var.index.isin(hvgs)].X.sum(axis=1).A1)
adata_orig = None

sc.pp.log1p(adata)
sc.pp.regress_out(adata, ['log_usable_counts'])
sc.pp.scale(adata)
sc.tl.pca(adata, zero_center=False, random_state=0)
sc.pp.neighbors(adata, n_neighbors=30, method='umap', metric='cosine', random_state=0, n_pcs=50)
sc.tl.leiden(adata, resolution=1.0, random_state=0)
sc.tl.umap(adata, min_dist=0.3, random_state=0)
      
sc.pl.umap(adata, color=['leiden'], size=9, legend_loc='on data', save = '{}_cluster.pdf'.format(sample_name))
sc.pl.umap(adata, color=['log_usable_counts'], size=6, color_map='Blues', title='log(read depth)')

fig, ax1 = plt.subplots(1,1,figsize=(7,5))
sns.boxplot(x='leiden', y='log_usable_counts', data=adata.obs)
plt.show()

fig, ax1 = plt.subplots(1,1,figsize=(7,5))
sns.boxplot(x='leiden', y='frac_reads_in_peaks', data=adata.obs)
plt.show()

#beta, alpha, delta
sc.pl.umap(adata, color=['INS','GCG','SST'], size=9, color_map='Blues', frameon=True, use_raw=True)
#gamma, ductal, acinar
sc.pl.umap(adata, color=['PPY','CFTR','REG1A'], size=9, color_map='Blues', frameon=True, use_raw=True)
#endothelial, immune, macrophage
sc.pl.umap(adata, color=['CDH5', 'CD69','C1QC'], size=9, color_map='Blues', frameon=True, use_raw=True)
#stellate, beta, alpha
sc.pl.umap(adata, color=['PDGFRB','PDX1','ARX'], size=9, color_map='Blues', frameon=True, use_raw=True)

sc.pl.dotplot(adata, ['INS','GCG','SST','PPY','CFTR','REG1A','CDH5', 'CD69','C1QC','PDGFRB','PDX1','ARX'], standard_scale='var', groupby='leiden', dendrogram=False, use_raw=True)

In [ ]:
adata

adata.obs.to_csv(os.path.join(CLEAN_DIR, '{}_clean_min1k.txt'.format(sample_name)), sep='\t')

In [ ]:
adata.obs

In [ ]:
#EDIT: PATH TO SAVE CLUSTERING OUTPUT
clean = '/data/'
adata.obs.to_csv(os.path.join(clean, '{}_clean_min1k.txt'.format(sample_name)), sep='\t')
